# Lesson 78 · Beat tracking from scratch — teaching an AI to nod along to music: the onset envelope, autocorrelation and BPM estimation

**Learning goals**
1. Understand how onset (note start) detection works: spectral flux
2. Implement onset_envelope (pure NumPy + aurora.audio.stft)
3. Estimate the beat period and BPM with autocorrelation
4. Check your result against aurora.music.beat_track

> The onset curve in plain words: "energy suddenly jumps up"; rubato or changing tempo will break it — note where the method stops working.

← **Previous lesson** · [L77 · Music features](L77_music_features.ipynb)

> Last lesson covered **music features**: chroma, RMS energy and ZCR, implemented from scratch in aurora.music.
> This lesson explores **beat tracking from scratch**.

## This lesson's story: how does an AI catch the strong beats that keep coming back in music?

Clap out a steady rhythm and people start nodding along — because music has regular **bursts of energy**: drum hits, plucked strings, breaths. The brain is hunting for these "moments when the energy suddenly jumps", which physics calls **onsets**.

**The core idea: two stages**

```
audio signal
   ↓ STFT → magnitude spectrum (per frame)
   ↓ spectral flux = sum of the positive differences between neighboring frames' magnitude spectra
   ↓ onset_envelope (one value per frame: large = a strong beat, small = quiet)
   ↓ autocorrelation to find the period
   ↓ peak lag → BPM = 60 / (lag × hop / sr)
```

**Spectral flux**:
```
SF[t] = sum(max(|X[t]| - |X[t-1]|, 0))  summed over frequency bins (magnitude-spectrum difference, matching aurora.music)
```
Only positive differences count — a sudden rise in energy is an onset; energy dying away is not.

**Finding the period with autocorrelation**:
```
R[τ] = sum(envelope[t] × envelope[t+τ])  summed over all t
```
If the beat period is T frames, R has peaks at τ=T, 2T, 3T. Find the biggest peak → BPM.

**Your task in this section**: implement `my_onset_envelope()` and `my_beat_track()` in pure NumPy, and check them on a synthetic 120 BPM signal.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.stft import stft

In [ ]:
# Generate a test signal: simulated drum hits at 120 BPM
SR = 16000
DURATION = 4.0
HOP = 256
N_FFT = 2048

t = np.linspace(0, DURATION, int(SR * DURATION), endpoint=False)

# Simulated drum hits: a short burst every 0.5 seconds
beat_period = SR // 2   # 120 BPM
signal = np.zeros(len(t))
for beat_start in range(0, len(t), beat_period):
    end = min(beat_start + SR // 50, len(t))   # 20 ms burst
    signal[beat_start:end] = np.random.randn(end - beat_start) * 0.8

signal = signal / (np.abs(signal).max() + 1e-8)
print(f'Signal length: {len(signal)} samples ({DURATION}s @ {SR}Hz)')
print(f'Expected BPM: 120, expected beat period: {beat_period} samples = {beat_period/SR*1000:.0f}ms')

## ✏️ Task 1: implement onset_envelope

**Signature**:
```python
def my_onset_envelope(signal, sample_rate, n_fft=N_FFT, hop_length=HOP) -> np.ndarray:
    # returns: (n_frames,) float, the spectral-flux envelope
```

**A 4-step route to the implementation**:

| Step | Operation | Tool |
|---|---|---|
| 1 | STFT → magnitude spectrum | `np.abs(stft(signal, n_fft=n_fft, hop_length=hop_length))` |
| 2 | Difference between neighboring frames' magnitude spectra | `np.diff(mag, axis=0)` → shape (n_frames-1, n_bins) |
| 3 | Keep only positive differences (energy increases) | `np.maximum(diff, 0)` |
| 4 | Sum each frame | `.sum(axis=1)` → (n_frames-1,) |

**Acceptance criteria**:
- Drum-hit signal (120 BPM): the envelope has clear peaks at the drum hits
- Output shape = `(n_frames-1,)` (one less than the number of frames, because of the diff)
- All-zero signal → envelope all 0

In [ ]:
def my_onset_envelope(signal, sample_rate, n_fft=N_FFT, hop_length=HOP):
    """Spectral-flux onset envelope: sum of positive magnitude-spectrum differences between neighboring frames."""
    # stft returns shape (n_frames, n_fft//2+1); take the difference along axis=0 (time) and sum along axis=1 (frequency)
    # ✏️ TODO: call stft, take the absolute value, take the difference, half-wave rectify, sum over frequency
    raise NotImplementedError("TODO")

try:
    env = my_onset_envelope(signal, SR)
    fps = SR / HOP
    times = np.arange(len(env)) / fps
    print(f'onset envelope shape: {env.shape}')
    print(f'Frame rate: {fps:.1f} fps')
    # ✅ check the shape and value range
    assert env.ndim == 1, "env should be a 1D array"
    assert (env >= 0).all(), "spectral flux must be non-negative (should be ≥ 0 after half-wave rectification)"
    n_frames = stft(signal, n_fft=N_FFT, hop_length=HOP).shape[0]
    assert env.shape[0] == n_frames - 1, (
        f"env length should be n_frames-1={n_frames-1}, got {env.shape[0]}; "
        "reason: np.diff along the time axis loses one frame"
    )
    print('✅ Task 1 checks passed')
    plt.figure(figsize=(10, 2))
    plt.plot(times, env)
    plt.xlabel('Time (s)')
    plt.title('Onset envelope (spectral flux)')
    plt.tight_layout()
    plt.show()
except (NotImplementedError, TypeError):
    print('⚠️  Implement my_onset_envelope first, then run this cell.')
    env, times = None, None
    fps = SR / HOP


### 🤔 Think first: how do you "count" beats from the envelope?

Don't look at the formula yet. Suppose the envelope "bumps" every T frames, but you **don't know T**. How would you find it?

A brute-force idea: **make a copy of the envelope, shift it right by τ frames**, then multiply it point by point with the original and add everything up — that is the autocorrelation `R[τ]`.

- When τ happens to equal the beat period T: drum hits line up with drum hits → the sum of products is largest (a peak).
- When τ doesn't line up: drum hits land on quiet spots → the sum of products is small.

So the one-line intuition: **autocorrelation = how similar a signal is to a delayed copy of itself; the position of the similarity peak = the beat period.** Find where that peak is, convert it to BPM, and you're done. Task 2 below turns that sentence into code.

## ✏️ Task 2: estimate BPM with autocorrelation

**Signature**:
```python
def my_beat_track(signal, sample_rate, hop_length=HOP) -> tuple[float, np.ndarray]:
    # returns: (bpm: float, beat_times: np.ndarray)
    # beat_times: in seconds, the time position of each beat
```

**A 3-step route to the implementation**:

| Step | Operation | Key formula |
|---|---|---|
| 1 | Call `my_onset_envelope(signal, sample_rate)` to get the envelope | shape (n_frames-1,) |
| 2 | Autocorrelation: `R[τ] = sum(envelope × envelope[τ:])` | dot product lag by lag: `np.dot(env[:-lag], env[lag:])` (equivalent to the second half of `np.correlate(e, e, 'full')`; written by hand here so the definition stays visible) |
| 3 | Find the largest lag within a sensible BPM range (40–240, matching the code) | `lag_samples = argmax(R[min_lag:max_lag])` (the lag is counted in frames) |

**BPM conversion**:
```
lag_in_frames = argmax(R)
lag_in_seconds = lag_in_frames × hop_length / sample_rate
BPM = 60.0 / lag_in_seconds
```

**Acceptance criteria**:
- 120 BPM synthetic signal → BPM estimate off by < 15 BPM (matching `abs(bpm - 120) < 15` in the check cell below)
- `beat_times` is in seconds, and the first beat falls in a sensible range

In [ ]:
def my_beat_track(signal, sample_rate, hop_length=HOP):
    """Estimate the BPM and beat times from the autocorrelation."""
    env = my_onset_envelope(signal, sample_rate, hop_length=hop_length)
    fps = sample_rate / hop_length

    # lag range (in frames) corresponding to the BPM range 40-240
    min_lag = max(1, int(fps * 60 / 240))
    max_lag = min(len(env) - 1, int(fps * 60 / 40))
    lags = np.arange(min_lag, max_lag + 1)

    # ✏️ TODO — implement in four steps:
    # 1. compute the autocorrelation vector (dot product of env with shifted env):
    #    ac = [np.dot(env[:-lag], env[lag:]) for lag in lags]
    # 2. find the best lag (where the autocorrelation is largest):
    #    best_lag = lags[np.argmax(ac)]
    # 3. convert to BPM (formula: BPM = 60 × fps / lag):
    #    bpm = fps * 60.0 / best_lag
    # 4. find the beat times (seconds): find the phase of the first beat, then step forward by best_lag
    #    pos = np.argmax(env[:best_lag])              # frame index of the first beat
    #    beat_frames = np.arange(pos, len(env), best_lag)
    #    beats = beat_frames / fps                    # convert to seconds
    #    return bpm, beats
    raise NotImplementedError("TODO")

try:
    bpm, beats = my_beat_track(signal, SR)
    print(f'Estimated BPM: {bpm:.1f}  (expected: 120)')
    print(f'Number of beats: {len(beats)}  (reference: ~{int(DURATION * 2)})')
    assert abs(bpm - 120) < 15, f'BPM is too far off: {bpm:.1f}'
    print('✅ Beat tracking checks passed')
except (NotImplementedError, TypeError):
    print('⚠️  Implement my_beat_track first, then run this cell.')
    bpm, beats = None, None


In [ ]:
# Compare with aurora.music
# Note: the reference onset_envelope / beat_track both default to n_fft=2048; this lesson uses the same settings
from aurora.music import onset_envelope, beat_track

ref_env = onset_envelope(signal, SR, n_fft=N_FFT, hop_length=HOP)
ref_bpm, ref_beats = beat_track(signal, SR, hop_length=HOP)

if bpm is not None:
    print(f'Your BPM:         {bpm:.1f}')
    print(f'aurora.music BPM: {ref_bpm:.1f}')
    print(f'Difference: {abs(bpm - ref_bpm):.1f} BPM')
else:
    print(f'aurora.music reference BPM: {ref_bpm:.1f}  (compare once implemented)')


## Wrap-up

| Concept | Remember |
|---|---|
| Spectral flux | sum of positive differences; detects sudden rises in energy |
| Autocorrelation | maximum at lag=τ → period τ → BPM |
| aurora.music | beat_track() is the reference implementation of the algorithm you just wrote |
| L79 | with chroma + RMS + BPM features in hand, you can build music embedding vectors |

The next lesson (L79) trains music embeddings with contrastive learning: a triplet loss pulls tracks of the same kind together in vector space and pushes different kinds apart.

## ✏️ Whiteboard challenge: beat tracking by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: The spectral flux formula: SF[t] = ? Why keep only the positive differences?
(SF[t] = Σ max(|X[t,k]|-|X[t-1,k]|, 0), a magnitude-spectrum difference; keeping only positive differences → detect only energy increases (onsets); energy decay (offsets) isn't a beat)

**Question 2**: For music at 120 BPM, with hop=256 and sr=16000, how many frames does one beat span?
(beat period = 60/120 = 0.5 seconds; 0.5×16000/256 = 31.25 ≈ 31 frames)

**Question 3**: What does the autocorrelation R[τ] mean physically? Why is there a peak at the beat period?
(R[τ] = the dot product of the envelope with itself shifted by τ; if the beat period = T, there is an energy peak every T frames, so after shifting by T the peaks line up → R[T] is large)

**Question 4**: If the BPM estimate comes out as 60 (when it is really 120), what is the most likely cause?
(The autocorrelation peak at τ=2T may be bigger than the one at τ=T — the tempo octave error (octave ambiguity); a subharmonic gets mistaken for the fundamental period)

**Question 5**: What happens to the BPM estimate when onset_envelope is all zeros? How should it be handled?
(R[τ]=0 for every τ, so argmax just returns the first index — a meaningless lag: in this lesson's code that is min_lag, i.e. a bogus 240 BPM, and if lag 0 were included you'd divide by zero / get inf BPM; check envelope.max() > threshold, and return nan or raise an error if it is 0)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

SR_TEST = 16000
HOP_TEST = 256
BPM_TRUE = 120.0

# Q1: spectral flux concept check (build frames with rising vs falling energy)
power_prev = np.array([1.0, 2.0, 3.0])
power_curr = np.array([3.0, 1.0, 5.0])
sf = np.sum(np.maximum(power_curr - power_prev, 0))
assert sf == 4.0, f"SF should be max(2,0)+max(-1,0)+max(2,0)=4, got {sf}"
print(f"Q1 ✅  spectral flux: max(2,0)+max(-1,0)+max(2,0)={sf} (keep energy increases only, ignore decay)")

# Q2: frames per beat at 120 BPM
beat_period_sec = 60.0 / BPM_TRUE
beat_period_frames = beat_period_sec * SR_TEST / HOP_TEST
expected_frames = 31.25  # about 31 frames
assert abs(beat_period_frames - expected_frames) < 0.1, f"frames per beat={beat_period_frames:.2f}, expected ≈{expected_frames}"
print(f"Q2 ✅  120BPM: beat={beat_period_sec}s=0.5×{SR_TEST}/{HOP_TEST}={beat_period_frames:.2f} frames")

# Q3: autocorrelation demo
np.random.seed(42)
n_frames = 120
# simulated 120 BPM envelope: one pulse every ~31 frames
envelope_test = np.zeros(n_frames)
for i in range(0, n_frames, int(beat_period_frames)):
    if i < n_frames:
        envelope_test[i] = 1.0
corr = np.correlate(envelope_test, envelope_test, 'full')
half = corr[n_frames-1:]  # positive-lag part
peak_lag = np.argmax(half[5:]) + 5  # skip lag=0 (largest, but meaningless)
bpm_est = 60 / (peak_lag * HOP_TEST / SR_TEST)
print(f"Q3 ✅  autocorrelation demo: peak_lag={peak_lag} frames, estimated BPM={bpm_est:.1f} (theory={BPM_TRUE})")

# Q4: tempo octave error demo
corr_at_T = half[int(beat_period_frames)]
corr_at_2T = half[int(2*beat_period_frames)]
print(f"Q4 ✅  R[T={int(beat_period_frames)}]={corr_at_T:.2f}, R[2T={int(2*beat_period_frames)}]={corr_at_2T:.2f} — if the 2T peak is bigger, the estimate is 60 BPM (tempo octave error)")

# Q5: handling an all-zero envelope
try:
    zeros = np.zeros(SR_TEST * 2)
    env_zeros = my_onset_envelope(zeros, SR_TEST)
    assert np.all(env_zeros == 0), f"onset_envelope of an all-zero signal should be all 0"
    print(f"Q5 ✅  all-zero signal: onset_envelope is all 0 (beat_track should check for and handle this)")
except (NotImplementedError, TypeError):
    print(f"Q5 ✅  all-zero envelope → BPM cannot be estimated; return nan when envelope.max()<threshold (will be verified once implemented)")

print("\n🎉 Beat-tracking whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l78_review = {
    "spectral_flux_understood":  None,  # understand spectral flux = positive differences only (energy increases)? True/False
    "autocorr_bpm_logic":        None,  # understand finding the period with autocorrelation: R[T] large → beat period = T frames → BPM=60/(T×hop/sr)? True/False
    "onset_envelope_impl":       None,  # my_onset_envelope() implemented correctly, clear peaks on the 120 BPM signal? True/False
    "beat_track_impl":           None,  # my_beat_track() implemented correctly, 120 BPM estimate off by < 15 BPM? True/False
    "whiteboard_passed":         None,  # all 5 whiteboard questions worked by hand? True/False
}

unfilled = [k for k, v in l78_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l78_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L78 complete! On to L79: music embeddings')

---

→ **Next lesson** · [L79 · Music embeddings](L79_music_embed.ipynb)

> Next lesson covers **music embeddings**: contrastive learning — pull similar styles together, push genres apart.